# Conditional retrieval and local-model qualification sweep

This notebook creates a **bounded, conditional experiment design** and optionally qualifies a small number of model runtimes. Its default run is deliberately affordable: it schedules a 24-configuration mixed screen and runs one six-query MiniLM runtime smoke test. It does **not** execute a retrieval benchmark, establish accuracy, or prove that a model can route real package capabilities.

Every generated file is a receipt. Scheduled configurations are labeled `scheduled`; model checks are labeled `runtime_smoke`; failures are retained. A full benchmark requires a frozen package corpus, reviewed queries and relevance judgments, hard negatives and no-route tasks, repeated latency measurements, and verified recipe execution.

Turn on Internet in Kaggle. A GPU is optional for the default run and recommended only for the opt-in generator arms. Gemma 4 access may require accepting its license and adding `HF_TOKEN` as a Kaggle secret. The Ollama arm only contacts a server that you have already started; this notebook never installs Ollama or pulls a model.

In [ ]:
# Small bootstrap only. Model libraries are installed later only for enabled arms.
!pip install -q "packaging>=24" "numpy>=1.26" "huggingface-hub>=0.28"

## Editable settings

`smoke` and `screen` are work budgets, not quality labels. Keep the defaults for a first run. Enable only one new model at a time on a 16 GB T4 so failures and peak-memory receipts remain attributable. Embedding models run sequentially and are unloaded between arms. Disabled examples are included for EmbeddingGemma and Qwen3 Embedding.

In [ ]:
from collections import Counter
from datetime import datetime, timezone
from importlib import metadata as importlib_metadata
from importlib import resources
from pathlib import Path
from typing import Any
import gc
import importlib
import json
import os
import re
import subprocess
import sys

OUTPUT_DIR = Path("/kaggle/working/retrieval_model_sweep")
REPOSITORY_URL = "https://github.com/Amarel-Taylor-Scott/this-already-exists-dont-rebuild-it.git"
REPOSITORY_REF = "main"
BOOTSTRAP_REPOSITORY_IF_NEEDED = True

DESIGN_STRATEGY = "mixed_screen"
MAX_EXPERIMENTS = 24
DESIGN_SEED = 17
MAX_RESOURCE_TIER = "t4_light"  # cpu_small | cpu_large | t4_light | t4_full | beyond_t4
BUDGET_ID = "smoke"

RUN_EMBEDDING_QUALIFICATION = True
EMBEDDING_QUALIFICATIONS = [
    {
        "enabled": True,
        "model_id": "sentence-transformers/all-MiniLM-L6-v2",
        "revision": None,
        "instruction": "none",
        "precision": "fp32",
        "truncate_dimension": None,
        "device": None,
        "trust_remote_code": False,
    },
    {
        "enabled": False,
        "model_id": "google/embeddinggemma-300m",
        "revision": None,
        "instruction": "model_default_asymmetric",
        "precision": "fp32",
        "truncate_dimension": 256,
        "device": "cpu",
        "trust_remote_code": False,
    },
    {
        "enabled": False,
        "model_id": "Qwen/Qwen3-Embedding-0.6B",
        "revision": None,
        "instruction": "natural_language_to_code",
        "precision": "fp16",
        "truncate_dimension": None,
        "device": "cuda",
        "trust_remote_code": False,
    },
]

RUN_TRANSFORMERS_GEMMA4 = False
GEMMA4_MODEL_ID = "google/gemma-4-E2B-it"
GEMMA4_REVISION = None
GEMMA4_QUANTIZATION = "bitsandbytes_nf4"  # fp16 | bitsandbytes_int8 | bitsandbytes_nf4
GEMMA4_THINKING = False
GEMMA4_MAX_NEW_TOKENS = 192
GEMMA4_CHECK_CACHE_PARITY = False  # Enable later; it performs a second full load.
HF_TOKEN_SECRET_NAME = "HF_TOKEN"

RUN_OLLAMA_EXISTING_SERVER = False
OLLAMA_BASE_URL = "http://127.0.0.1:11434"
OLLAMA_MODEL_TAG = "gemma4:e2b"
OLLAMA_THINKING = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR = OUTPUT_DIR / "model_cache"
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

SETTINGS = {
    "classification": "smoke_configuration",
    "design_strategy": DESIGN_STRATEGY,
    "max_experiments": MAX_EXPERIMENTS,
    "design_seed": DESIGN_SEED,
    "max_resource_tier": MAX_RESOURCE_TIER,
    "budget_id": BUDGET_ID,
    "run_embedding_qualification": RUN_EMBEDDING_QUALIFICATION,
    "enabled_embedding_models": [
        item["model_id"] for item in EMBEDDING_QUALIFICATIONS if item["enabled"]
    ],
    "run_transformers_gemma4": RUN_TRANSFORMERS_GEMMA4,
    "run_ollama_existing_server": RUN_OLLAMA_EXISTING_SERVER,
    "benchmark_claims_permitted": False,
}
print(json.dumps(SETTINGS, indent=2))

## Bootstrap the repository and only the enabled model dependencies

The repository may be the current checkout, a Kaggle input, or a shallow clone under `/kaggle/working`. It is installed as a wheel so the versioned experiment registry is loaded as packaged data. The recorded Git commit and installed distribution version make the design reproducible.

In [ ]:
def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, indent=2, sort_keys=True, default=str) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)

def find_project_root() -> Path | None:
    candidates: list[Path] = []
    configured = os.getenv("REUSE_REPO_DIR", "").strip()
    if configured:
        candidates.append(Path(configured))
    candidates.extend([Path.cwd(), Path.cwd().parent])
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        candidates.extend(item for item in kaggle_input.iterdir() if item.is_dir())
    seen: set[Path] = set()
    for candidate in candidates:
        for root in (candidate, candidate / "project"):
            root = root.resolve()
            if root in seen:
                continue
            seen.add(root)
            if (root / "pyproject.toml").is_file() and (
                root / "src" / "existing_code_reuse"
            ).is_dir():
                return root
    return None

PROJECT_ROOT = find_project_root()
CLONE_DIR = Path("/kaggle/working/this-already-exists-dont-rebuild-it")
if PROJECT_ROOT is None and BOOTSTRAP_REPOSITORY_IF_NEEDED:
    if not CLONE_DIR.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                REPOSITORY_REF,
                REPOSITORY_URL,
                str(CLONE_DIR),
            ],
            check=True,
        )
    PROJECT_ROOT = CLONE_DIR
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Repository not found. Attach it, set REUSE_REPO_DIR, or enable bootstrap."
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", str(PROJECT_ROOT)],
    check=True,
)
enabled_embeddings = [item for item in EMBEDDING_QUALIFICATIONS if item["enabled"]]
model_dependencies: list[str] = []
if RUN_EMBEDDING_QUALIFICATION and enabled_embeddings:
    model_dependencies.append("sentence-transformers>=3.4")
if RUN_TRANSFORMERS_GEMMA4:
    model_dependencies.extend(
        ["transformers>=5.10.1", "accelerate>=1.2", "bitsandbytes>=0.45"]
    )
model_dependency_install = None
if model_dependencies:
    install_result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *model_dependencies],
        check=False,
        capture_output=True,
        text=True,
    )
    model_dependency_install = {
        "returncode": install_result.returncode,
        "stdout_tail": install_result.stdout[-4000:],
        "stderr_tail": install_result.stderr[-4000:],
    }
importlib.invalidate_caches()

if not os.getenv("HF_TOKEN") and (RUN_TRANSFORMERS_GEMMA4 or enabled_embeddings):
    try:
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret(HF_TOKEN_SECRET_NAME)
        if token:
            os.environ["HF_TOKEN"] = token
    except Exception:
        pass

git_commit = None
try:
    git_commit = subprocess.run(
        ["git", "rev-parse", "HEAD"],
        cwd=PROJECT_ROOT,
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
except Exception:
    pass

BOOTSTRAP_RECEIPT = {
    "classification": "environment_bootstrap_receipt",
    "captured_at": utc_now(),
    "project_root": str(PROJECT_ROOT),
    "git_commit": git_commit,
    "installed_version": importlib_metadata.version("existing-code-reuse"),
    "model_dependencies_requested": model_dependencies,
    "model_dependency_install": model_dependency_install,
}
write_json(OUTPUT_DIR / "bootstrap-receipt.json", BOOTSTRAP_RECEIPT)
print(json.dumps(BOOTSTRAP_RECEIPT, indent=2))

## Schedule a bounded conditional design

The registry contains independent and conditional factors for representations, deterministic features, NLP-generated candidates, blocking and LSH, sparse and dense retrieval, embedding instructions and dimensions, vector indexes, fusion, graph expansion, reranking, compatibility, planning runtimes, quantization, and verification. A literal Cartesian product is intentionally not run. The mixed screen retains baselines and named references, screens individual and pair interactions, and adds seeded valid combinations under the selected resource tier.

In [ ]:
from existing_code_reuse.experiments import (
    design_configurations,
    design_manifest,
    load_experiment_space,
    schedule_first_round,
)

registry_resource = resources.files("existing_code_reuse").joinpath(
    "experiment_space.json"
)
with resources.as_file(registry_resource) as registry_path:
    experiment_space = load_experiment_space(registry_path)

if BUDGET_ID not in experiment_space.budget_by_id:
    raise ValueError(
        f"Unknown budget {BUDGET_ID!r}; choose from {sorted(experiment_space.budget_by_id)}"
    )
configurations = design_configurations(
    experiment_space,
    strategy=DESIGN_STRATEGY,
    max_experiments=MAX_EXPERIMENTS,
    seed=DESIGN_SEED,
    max_resource_tier=MAX_RESOURCE_TIER,
)
trials = schedule_first_round(
    configurations,
    budget=experiment_space.budget_by_id[BUDGET_ID],
    registry_digest=experiment_space.registry_digest,
)
manifest = design_manifest(
    experiment_space,
    configurations,
    trials,
    strategy=DESIGN_STRATEGY,
    seed=DESIGN_SEED,
    max_resource_tier=MAX_RESOURCE_TIER,
    requested_configuration_count=MAX_EXPERIMENTS,
)
manifest.update(
    {
        "classification": "scheduled_smoke_design",
        "budget_id": BUDGET_ID,
        "created_at": utc_now(),
        "git_commit": git_commit,
        "benchmark_metrics_present": False,
    }
)
MANIFEST_PATH = OUTPUT_DIR / "experiment-design-manifest.json"
write_json(MANIFEST_PATH, manifest)
print(
    json.dumps(
        {
            "classification": manifest["classification"],
            "raw_cartesian_size": manifest["raw_cartesian_size"],
            "scheduled_configuration_count": manifest["scheduled_configuration_count"],
            "dimension_count": manifest["dimension_count"],
            "constraint_count": manifest["constraint_count"],
            "manifest_path": str(MANIFEST_PATH),
            "warning": manifest["warning"],
        },
        indent=2,
    )
)

## Record hardware before model work

The receipt records CUDA availability, exact GPU model, memory, compute capability, BF16 support, driver output, Python, and relevant library versions. Accelerator labels such as `T4` are observations, not assumptions.

In [ ]:
from existing_code_reuse.model_qualification import (
    hardware_snapshot,
    qualify_embedding_model,
    qualify_ollama_generator,
    qualify_transformers_generator,
)

HARDWARE_RECEIPT = hardware_snapshot()
HARDWARE_RECEIPT["classification"] = "measured_hardware_receipt"
write_json(OUTPUT_DIR / "hardware-receipt.json", HARDWARE_RECEIPT)
print(json.dumps(HARDWARE_RECEIPT, indent=2))

## Optional sequential embedding qualification

Each enabled model is loaded alone, encodes six synthetic capability queries and six matching descriptions, verifies finite nonzero vectors, records top-1 behavior, latency, revision, device and peak CUDA allocation, writes its vectors, and unloads. This is a wiring/runtime smoke test—not retrieval evidence. EmbeddingGemma is shown in FP32 on CPU because FP16 activations are not a safe qualification assumption; throughput batching belongs in a later, separate experiment.

In [ ]:
def safe_slug(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", value.casefold()).strip("-")[:96]

def preserved_failure(kind: str, identifier: str, error: Exception) -> dict[str, Any]:
    return {
        "classification": kind,
        "status": "failed",
        "identifier": identifier,
        "error_type": type(error).__name__,
        "error": str(error),
        "completed_at": utc_now(),
        "synthetic_smoke_only": True,
    }

qualification_receipts: list[dict[str, Any]] = []
if RUN_EMBEDDING_QUALIFICATION:
    for specification in EMBEDDING_QUALIFICATIONS:
        if not specification["enabled"]:
            continue
        model_id = str(specification["model_id"])
        receipt_path = (
            OUTPUT_DIR
            / "embedding_qualification"
            / safe_slug(model_id)
            / "receipt.json"
        )
        try:
            receipt = qualify_embedding_model(
                model_id,
                revision=specification["revision"],
                instruction=str(specification["instruction"]),
                precision=str(specification["precision"]),
                truncate_dimension=specification["truncate_dimension"],
                device=specification["device"],
                cache_dir=MODEL_CACHE_DIR,
                trust_remote_code=bool(specification["trust_remote_code"]),
                output_path=receipt_path,
            )
        except Exception as error:
            receipt = preserved_failure(
                "embedding_runtime_smoke", model_id, error
            )
            write_json(receipt_path, receipt)
        qualification_receipts.append(receipt)
        gc.collect()
        try:
            import torch

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass
        print(model_id, receipt.get("status"), receipt_path)
else:
    print("Embedding qualification skipped by setting.")

## Optional Gemma 4 Transformers qualification

When enabled, this arm loads `google/gemma-4-E2B-it` with the official Transformers multimodal model path, asks it to return a constrained JSON route over a fixed allowlist, and rejects unknown operations or malformed decisions. NF4 is the editable 16 GB GPU default. Cache parity is disabled for the first smoke because it requires another full model load. This tests only loading, generation and strict parsing; it does not measure planner accuracy.

In [ ]:
if RUN_TRANSFORMERS_GEMMA4:
    if not os.getenv("HF_TOKEN"):
        try:
            from kaggle_secrets import UserSecretsClient

            token = UserSecretsClient().get_secret(HF_TOKEN_SECRET_NAME)
            if token:
                os.environ["HF_TOKEN"] = token
        except Exception:
            pass
    gemma_receipt_path = OUTPUT_DIR / "gemma4_transformers" / "receipt.json"
    try:
        gemma_receipt = qualify_transformers_generator(
            GEMMA4_MODEL_ID,
            revision=GEMMA4_REVISION,
            quantization=GEMMA4_QUANTIZATION,
            max_new_tokens=GEMMA4_MAX_NEW_TOKENS,
            thinking=GEMMA4_THINKING,
            cache_dir=MODEL_CACHE_DIR,
            check_cache_parity=GEMMA4_CHECK_CACHE_PARITY,
            output_path=gemma_receipt_path,
        )
    except Exception as error:
        gemma_receipt = preserved_failure(
            "generator_runtime_smoke", GEMMA4_MODEL_ID, error
        )
        write_json(gemma_receipt_path, gemma_receipt)
    qualification_receipts.append(gemma_receipt)
    print(GEMMA4_MODEL_ID, gemma_receipt.get("status"), gemma_receipt_path)
else:
    print("Transformers Gemma 4 qualification skipped by setting.")

## Optional existing Ollama server qualification

This arm is off by default and contains no Ollama installer, daemon launch, or model pull. If you independently start a compatible local server and make the configured tag available, it sends the same allowlisted JSON-routing smoke request to `OLLAMA_BASE_URL`. Connection, missing-model, schema and generation failures are written as receipts instead of being hidden.

In [ ]:
if RUN_OLLAMA_EXISTING_SERVER:
    ollama_receipt_path = OUTPUT_DIR / "ollama_existing_server" / "receipt.json"
    try:
        ollama_receipt = qualify_ollama_generator(
            OLLAMA_MODEL_TAG,
            base_url=OLLAMA_BASE_URL,
            thinking=OLLAMA_THINKING,
            output_path=ollama_receipt_path,
        )
    except Exception as error:
        ollama_receipt = preserved_failure(
            "generator_runtime_smoke", OLLAMA_MODEL_TAG, error
        )
        write_json(ollama_receipt_path, ollama_receipt)
    qualification_receipts.append(ollama_receipt)
    print(OLLAMA_MODEL_TAG, ollama_receipt.get("status"), ollama_receipt_path)
else:
    print("Existing Ollama server qualification skipped by setting.")

## Write the smoke summary

The summary separates scheduling from measurement. `measured` here means a synthetic runtime check completed; it does not mean the corresponding configuration won a retrieval benchmark. Failed arms stay in the output and should be analyzed alongside successful arms.

In [ ]:
status_counts = Counter(str(item.get("status", "unknown")) for item in qualification_receipts)
SUMMARY = {
    "schema_version": "1",
    "classification": "runtime_smoke_summary",
    "completed_at": utc_now(),
    "git_commit": git_commit,
    "design_manifest": str(MANIFEST_PATH),
    "scheduled_configuration_count": len(configurations),
    "qualification_arm_count": len(qualification_receipts),
    "qualification_status_counts": dict(sorted(status_counts.items())),
    "qualification_receipts": [
        {
            "kind": item.get("qualification_kind", item.get("classification")),
            "model": item.get("model_id", item.get("model_tag", item.get("identifier"))),
            "backend": item.get("backend"),
            "status": item.get("status"),
            "synthetic_smoke_only": item.get("synthetic_smoke_only", True),
            "receipt_digest": item.get("receipt_digest"),
            "error": item.get("error"),
        }
        for item in qualification_receipts
    ],
    "benchmark_metrics_present": False,
    "benchmark_claims_permitted": False,
    "next_gate": (
        "Run a frozen reviewed retrieval benchmark with hard negatives, no-route tasks, "
        "an exact-search oracle, repeated latency measurements, and verified executions."
    ),
}
SUMMARY_PATH = OUTPUT_DIR / "runtime-smoke-summary.json"
write_json(SUMMARY_PATH, SUMMARY)
print(json.dumps(SUMMARY, indent=2))
print("Artifacts:", OUTPUT_DIR)

## What turns this into benchmark evidence

Do not rank models from the six synthetic smoke pairs. Promote a configuration only after it is evaluated on the same immutable corpus and reviewed task split. At minimum, record blocking recall against an unblocked exact-search oracle; Recall@k, nDCG@k, MRR, calibration and abstention on positive, hard-negative and no-route queries; index size, build time, cold/warm latency and peak memory; compatibility rejection accuracy; and execution outcomes from repeated isolated runs. Keep the holdout sealed until the retrieval and planner choices are frozen.